# LAB 6: Load Wizard 및 Snowflake Marketplace

👉 이 실습에서는 파일 로딩 방법과 해당 프로세스를 용이하게 하는 Snowflake 오브젝트 유형에 대해 학습합니다. 또한 데이터 및 데이터 제품을 위한 앱 스토어와 같은 Snowflake Marketplace에 대해 알아보겠습니다.

시작하기에 앞서, 이 실습 전반에서 사용할 **컨텍스트 정보**를 가져오겠습니다. 

- **Start** 버튼을 클릭하여 이 노트북을 활성화하세요.

- 다음 Python 셀을 실행하세요.


#### :warning: 이 노트북에 대해 새 세션이 시작될 때마다, 후속 셀에서 사용할 '변수'를 구성하기 위해 아래 셀을 다시 실행해야 합니다. :warning:


In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_DB'
print('현재 CONTEXT 정보:')
print('---------------------------------')
print(session)
print('현재 USER는 ' + user)

## INSERT 구문은 빨리 한계가 온다 📓 

### Vegetable_details 테이블 데이터

우리 팀의 비즈니스 분석가들이 Rooting Depth 컬럼을 한 글자로 줄이고 CSV(쉼표로 구분된 값) 파일로 출력하기로 결정했다고 가정해 보겠습니다. 이 변경 자체는 괜찮지만, 문제가 생길 조짐이 보입니다. 
- 문제가 보이시나요? 
- 왜 이 CSV를 로드할 수 없을까요? 
- 대부분의 로우에는 쉼표가 몇 개 있나요? 
- Hot Peppers 로우에는 쉼표가 몇 개 있나요? 

![Vegetable_details 테이블 데이터(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_veg_details_table_1.png)


### vegetable_details 테이블을 생성하세요. 🥋 

노트북 셀에서 작업할 때는 `USE` 구문으로 명시적으로 컨텍스트를 지정할 수 있음을 기억하세요. 예: `USE SCHEMA db_name.schema_name`

이것의 대안은 **전체 오브젝트 표기법**을 사용하는 것입니다. 이는 SQL에서 오브젝트를 참조할 때마다 오브젝트의 전체 경로(계층)를 사용한다는 의미입니다. 더 길지만 명확성을 높일 수 있습니다. 이 명명 규칙의 일반적인 구조는 `<database_name>.<schema_name>.<object_name>` 형식을 따릅니다.

💡 **힌트**: 만약 잘못된 데이터베이스나 스키마에 테이블을 생성한 경우, `ALTER TABLE - RENAME` 명령을 사용하여 이동하거나, 테이블을 Drop한 후 드롭 메뉴 컨텍스트 설정을 업데이트하고 재생성할 수 있습니다. 


In [ ]:
-- Vegetable_Details 테이블 생성

CREATE TABLE IF NOT EXISTS {{user}}_garden_plants.veggies.vegetable_details (
    plant_name VARCHAR(25),
    root_depth_code VARCHAR(1)
) 
DATA_RETENTION_TIME_IN_DAYS = 7;

## 파일에서 테이블 로우 로드하기 

### 데이터를 가져옵니다. 

`veggie_details_a_to_k_comma_opt_enclosed.csv` 및 `veggie_details_k_to_z_pipe.csv` 라는 두 개의 CSV(쉼표로 구분된 값) 파일이 **COMMON_DB.RESOURCES.CLASS_FILES** 스테이지에 업로드되었습니다. 앞서 생성한 **VEGETABLE_DETAILS** 테이블에 이 데이터를 로드하겠습니다. 

### 첫 번째 파일을 다운로드합니다. 🥋 

다음 Python 코드 셀을 실행하고 생성된 링크를 클릭하여 파일(Artichoke to Kale)을 다운로드하세요.


In [ ]:
import streamlit as st
snowpark_df = session.sql("SELECT GET_PRESIGNED_URL(@common_db.resources.course_files, 'veggie_details_a_to_k_comma_opt_enclosed.csv')")
collected_data = snowpark_df.collect()
st.write('다음 링크를 클릭하여 파일을 다운로드하세요:')
st.write(collected_data[0][0])

### 데이터 로드 대화 상자 단계 🥋 

1. Snowsight 오브젝트 브라우저에서 **VEGETABLE_DETAILS** 테이블을 찾습니다.
1. **새 탭에 표 세부 정보 열기** 아이콘을 클릭하여 새 탭에서 이 화면을 실행하세요.

![데이터 로드 대화 상자(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_open_table_details_1.png)

1. 오른쪽 상단의 파란색 `Load Data` 버튼을 클릭하세요.
1. 다음 대화 상자가 나타납니다.
1. 웨어하우스 사용을 묻는 메시지가 표시되면 자신의 동물 이름이 지정된 웨어하우스 **(animal)_WH**를 선택하세요.

![데이터 로드 대화 상자(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_load_data_1.png)

1. 파란색 **Browse** 버튼을 클릭하세요.
1. 로컬에 다운로드한`veggie_details_a_to_k_comma_opt_enclosed.csv` 파일을 찾아 선택하여 엽니다.
1. 그런 다음 대화 상자의 오른쪽 하단 모서리에 있는 파란색 **Next** 버튼을 클릭하세요.
1. 확장된 데이터 로드 대화 상자가 나타납니다.
1. 파일의 기본값 대부분은 그대로 사용하되, 왼쪽 **File format** 상자의 **View options** 드롭다운 화살표를 클릭하여 일부 옵션을 조정합니다.

![확장된 데이터 로드 대화 상자(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_load_data_2.png)

1. **Header**에는 'Skip first line'을 선택하세요.
1. **Field optionally enclosed by**에는 'Double quotes (default)'를 선택하세요.
1. 대화 상자의 오른쪽 하단 모서리에 있는 파란색 **Load** 버튼을 클릭하세요.

![데이터 로드 파일 옵션(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_load_data_3.png)

1. 모든 과정이 정상적으로 진행되면 다음과 같은 성공 메시지 대화 상자가 표시됩니다.
1. **View table details**를 클릭하고, 나타나는 페이지에서 **Data Preview** 옵션을 사용하여 테이블에 로드된 데이터를 확인하세요.

![데이터 로드 성공(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_load_data_success_1_v2.png)


### 데이터 로드 중 실수에서 복구하기 📓

실수로 같은 파일을 두 번 로드했고 다시 시작하고 싶은 경우, 테이블을 비우는 `TRUNCATE` 명령을 실행하세요. 

`TRUNCATE TABLE (animal)_GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS;`

그런 다음 로드 프로세스를 다시 시작하세요. 


## 파일에서 테이블 로우 로드하기

### 두 번째 파일을 로드합니다. 🥋 

다음 Python 코드 셀을 실행하고 생성된 링크를 클릭하여 두 번째 파일(Kohlrabi to Zucchini)을 다운로드하세요.


In [ ]:
snowpark_df = session.sql("SELECT GET_PRESIGNED_URL(@common_db.resources.course_files, 'veggie_details_k_to_z_pipe.csv')")
collected_data = snowpark_df.collect()
st.write('다음 링크를 클릭하여 파일을 다운로드하세요:')
st.write(collected_data[0][0])

## 챌린지 실습: 파일을 테이블에 로드하기 🎯 

앞서 사용한 것과 동일한 **Load Data** 방식으로, 다운로드한 파일을 같은 테이블에 로드하세요. 

💡 **참고**: 이 파일을 로드할 때 첫 번째 파일과 비교하여 **View options** 중 최소 한 가지 설정이 다를 것입니다. 이번에는 컬럼이 쉼표로 구분되지 **않습니다**. 많은 경우에, Snowflake는 로드되는 파일의 형태를 자동으로 감지하고, 올바르게 로드될 수 있도록 옵션을 제안할 수 있습니다. 두 번째 파일을 로드할 때도 이런 기능이 작동하나요?

![식물 세부 정보 테이블 데이터(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_veg_details_csv_1.png)

💡 **팁**: 컬럼과 로우를 구분하는 문자를 확인하려면 Excel로 CSV 파일을 열지 마세요. 간단한 텍스트 편집기를 사용하세요. Windows에서는 메모장이 잘 작동합니다. Mac에서는 텍스트 편집기가 잘 작동합니다. 

파일이 로드되면 `SELECT` 구문을 실행하고 출력 결과를 검토하여 테이블을 확인하세요. 42개의 로우가 표시되어야 합니다. 정렬 순서를 반대로 하면(출력 테이블의 **PLANT_NAME** 컬럼 헤더를 클릭하면) 맨 위에 Zucchini가 표시되어 두 번째 파일을 로드했음을 확인할 수 있습니다.


In [ ]:
SELECT * 
FROM {{user}}_garden_plants.veggies.vegetable_details;

## 테이블 데이터 보기 

### SQL을 사용하여 vegetable_details 테이블을 확인하세요. 

이 챌린지 실습은 단계별 세부 사항을 포함하지 않으며, 여러 목표를 달성하기 위한 일반적인 지침만 제공합니다.

1. 데이터를 확인하세요.
1. 'Spinach'에 해당하는 레코드를 분리하세요.
1. 데이터 세트에 두 번 나타나는 식물 이름을 확인하세요.
1. 이를 제거할 방법을 찾으세요.
1. 데이터를 다시 확인하세요.

💡 **팁**: 아래 준비된 템플릿 쿼리에서 해시 문자(`#`)를 **교체**하세요. 이 문자가 있는 곳에는 사용자의 입력이 필요합니다. 

💡 **팁**: 정말 막히면 각 템플릿 SQL 구문 아래의 힌트 셀을 **확장**하세요. 셀 오른쪽 상단의 회색 재생 버튼 바로 옆에 있는 **View display options** 선택기에서 **Code and results**를 클릭하면 됩니다. 먼저 스스로 해결해 보려고 노력하세요 :grinning:.


### 1\. 데이터를 확인하세요. 🎯 


In [ ]:
-- 다음 코드를 수정하고 실행하여 먼저 컨텍스트(데이터베이스 및 스키마)를 설정한 후 모든 테이블 데이터를 확인하세요
USE DATABASE ######_######_######;
USE ###### veggies;

SELECT #
FROM #########_details;

In [ ]:
/*
-- 변수 치환 대신 다음 USE 문을 하드 코딩할 수도 있습니다
USE SCHEMA {{user}}_garden_plants.veggies;

SELECT *
FROM vegetable_details;
*/

### 2\. **plant_name**이 **Spinach**인 모든 로우를 반환하는 SQL 쿼리를 작성하세요. 🎯 

[`UPPER()`](https://docs.snowflake.com/ko/sql-reference/functions/upper) 함수를 사용할 것입니다. 이는 우리가 검사할 테스트 문자열을 `spinach`, `Spinach` 또는 기타 변형으로 작성할 필요가 없음을 의미합니다. 입력값과 컬럼에 저장된 값은 `SPINACH`로 변환됩니다.

💡 **힌트**: Spinach 로우가 두 개 있는 것을 보고 놀랐습니다.


In [ ]:
SELECT *
FROM #########_#######
WHERE UPPER(plant_name) = UPPER('#######');

In [ ]:
/*
SELECT *
FROM vegetable_details
WHERE UPPER(plant_name) = UPPER('Spinach');
*/

### 3\. 한 로우에는 얕은 뿌리를 나타내는 'S'가, 다른 로우에는 깊은 뿌리를 나타내는 'D'가 있습니다. 🎯 

Spinach 뿌리가 깊다고 표시된 로우를 제거해야 합니다. 먼저, 'D' 값이 포함된 로우만 따로 골라내 봅시다. 


In [ ]:
SELECT *
FROM vegetable_details
WHERE UPPER(plant_name) = UPPER('#######')
AND root_depth_code = '#';

In [ ]:
/*
SELECT *
FROM vegetable_details
WHERE UPPER(plant_name) = UPPER('Spinach')
AND root_depth_code = 'D';
*/

### 4\. ROOT_DEPTH_CODE 컬럼에 'D'가 있는 Spinach 로우만 제거하세요. 🎯 


In [ ]:
DELETE
FROM vegetable_details
WHERE UPPER(plant_name) = UPPER('#######')
AND root_depth_code = '#';

In [ ]:
/*
DELETE
FROM vegetable_details
WHERE UPPER(plant_name) = UPPER('Spinach')
AND root_depth_code = 'D';
*/

### 5\. 모든 데이터를 다시 확인하고, 두 번 나타나는 식물 이름이 없는지 확인하세요. 🎯 


In [ ]:
SELECT #
FROM #########_#######;

In [ ]:
/*
SELECT *
FROM vegetable_details;
*/

### File Format 오브젝트에 대한 참고 사항 📓

이 실습에서 여러분은 Snowsight의 Load Data Wizard를 사용하여, Snowflake가 파일을 읽고 그 데이터를 테이블에 삽입하도록 안내하는 과정을 경험했습니다. 보셨듯이, 이 대화 상자에는 Snowflake가 소스 파일에 있는 데이터의 “형태”를 이해하고, 파일의 데이터를 어떻게 해석하고 처리할지를 지정할 수 있는 옵션이 포함되어 있습니다. 여기에는 파일의 종류(예: CSV), 필드 구분자, 건너뛸 헤더 라인 포함 여부 등이 포함됩니다.

이러한 일련의 지침들은 [FILE FORMAT](https://docs.snowflake.com/ko/sql-reference/sql/create-file-format)이라고 불리는 Snowflake 오브젝트로 묶어서 관리할 수 있습니다. 이 오브젝트는 데이터 로드 대화 상자에서 참조하거나, [`COPY INTO`](https://docs.snowflake.com/ko/sql-reference/sql/copy-into-table) 명령을 통해 프로그래밍 방식으로 데이터를 로드할 때 사용할 수 있습니다. 동일한 사양으로 파일을 반복 로드할 때 이러한 지침을 재입력하거나 재코딩할 필요가 없어 시간을 절약할 수 있습니다.


## Cloning에 대한 간단한 소개 📓

Snowflake의 [Zero-Copy Cloning](https://docs.snowflake.com/ko/user-guide/object-clone) 기능은 모든 테이블, 스키마 또는 데이터베이스(및 기타 오브젝트)의 '스냅샷'을 빠르게 찍고 처음에 기초 스토리지를 공유하는 오브젝트의 파생 사본을 생성할 편리한 방법을 제공합니다. 이는 기초 데이터의 물리적 복사를 수반하지 않으며 메타데이터에만 영향을 미치는 작업입니다. 따라서 테이블, 스키마, 심지어 전체 데이터베이스의 Cloning은 일반적으로 매우 빠르게 수행됩니다.

Clone된 오브젝트는 생성되는 순간 독립적인 오브젝트가 됩니다. 소스 오브젝트에 수행하는 것과 동일한 작업을 Clone된 오브젝트에도 수행할 수 있습니다. 예를 들어, Clone된 테이블에는 데이터를 기록하거나 `ALTER TABLE` 명령을 사용하여 파라미터를 변경하는 등 모든 작업을 수행할 수 있습니다.

Cloning은 (Clone된 오브젝트에 변경 사항을 적용할 때까지) 추가 비용이 발생하지 않는 즉각적인 백업을 생성하는 데 매우 유용할 수 있습니다. Snowflake의 이러한 기능은 개발(Dev) 및 테스트 또는 QA 환경을 빠르게 프로비저닝하거나 데이터를 백업할 때 자주 사용됩니다.


### 테이블을 Clone합니다. 🥋

다음과 같이 **(animal)_DB** 데이터베이스 내에 **CLONED_OBJECTS**라는 새 스키마를 생성합니다. 그런 다음 기존 **VEGETABLE_DETAILS** 테이블을 이 새 스키마에 **VEGETABLE_DETAILS_CLONE**이라는 이름으로 `CLONE`합니다.

다음 코드를 실행하여 이 단계를 수행하고 작업 속도를 확인하세요. Snowflake에서 Cloning을 할 때는 테이블 데이터가 아니라 메타데이터(데이터에 대한 정보)만 복사된다는 점을 기억하세요.


In [ ]:
CREATE SCHEMA IF NOT EXISTS {{user}}_db.cloned_objects;

USE SCHEMA {{user}}_db.cloned_objects;

CREATE TABLE IF NOT EXISTS vegetable_details_clone
    CLONE {{user}}_garden_plants.veggies.vegetable_details;

### Clone된 테이블의 세부 정보를 확인해 봅시다.

`SHOW` 명령을 사용하여, Clone된 테이블과 그 소스 테이블을 함께 검토해 보세요. 두 테이블은 독립적인 오브젝트입니다. 단, 두 테이블 모두 로우 수와 바이트 수는 동일하지만, 스토리지 사용량은 소스 테이블만을 기준으로 측정됩니다.


In [ ]:
SHOW TABLES LIKE 'vegetable_details%' IN ACCOUNT;

### Clone된 테이블을 쿼리하세요.

**VEGETABLE_DETAILS**에서 했던 것처럼, Clone된 테이블 **VEGETABLE_DETAILS_CLONE**을 쿼리해 보세요. 익숙한 데이터가 보일 것입니다. 이 Clone된 오브젝트는 표준 Snowflake 테이블로, 생성 방식과 상관없이 모든 표준 테이블 작업을 지원합니다. 


In [ ]:
SELECT * 
FROM vegetable_details_clone;

## Snowflake Marketplace 📓

### Snowflake Marketplace 정의

Snowflake Marketplace는 Snowflake 계정 내에서 액세스 가능한 통합 포털입니다. Snowflake Marketplace를 사용하면 서드 파티 데이터와 서비스를 탐색하고 이용할 수 있으며, 자신의 데이터 제품을 Snowflake AI Data Cloud를 통해 판매할 수도 있습니다. 소비자 관점에서는 Snowflake Marketplace에서 제공되는 데이터를 활용하여 다음에 액세스할 수 있습니다.

- 연구, 예측, 머신러닝을 위한 과거 데이터
- 현재 날씨, 교통 상황과 같은 실시간 스트리밍 데이터
- 구독자 및 타깃 고객을 이해하기 위한 특화된 ID 데이터
- 예상치 못한 출처에서 얻는 새로운 인사이트 

### Snowflake에서 데이터를 테이블로 로드하는 것보다 쉬운 방법이 있을까요? 있습니다. 로드하지 않는 것입니다.

Snowflake Marketplace를 이용하면 점점 규모가 커지고 있는 글로벌 제공자가 제공하는 다양한 카테고리에서 비즈니스와 관련된 데이터와 데이터 제품을 검색할 수 있으며, 버튼 클릭만으로 계정 내에서 이를 활용할 수 있습니다. 수백 개의 데이터 목록이 있으며, **무료 또는 무제한 액세스** 옵션도 포함됩니다. Marketplace 거래를 수행할 수 있는 권한이 있다면, 공유 데이터는 몇 초 만에 계정에 새로운 데이터베이스로 나타납니다. 이를 분석하고 활용하며, 기존 데이터와 조인해 데이터를 확장하고 향상시키거나 새로운 데이터 제품을 만드는 데 활용할 수 있습니다. 데이터 로드는 필요 없습니다.

--- 

![Snowflake Marketplace(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_marketplace_v2.png)


### Snowflake Marketplace로 이동합니다. 🥋 

1. 왼쪽 메인 메뉴에서 **Work with data** > **Marketplace**를 선택합니다.
1. Snowflake Marketplace 검색 창을 사용해 'books'를 입력하고 Enter 키를 누릅니다.
1. 표시되는 하위 메뉴에서 **Pricing** > **Free**를 선택합니다.
1. **AI Training Dataset from Goodreads Books** Listing을 찾습니다.
1. 링크를 클릭하고 데이터 사전 사양 및 사용 예시(SQL 쿼리)를 포함한 이 데이터 세트의 세부 정보를 검토하세요.

💡 **팁**: Snowflake Marketplace Listing을 이용하려면 계정의 특별 권한이 필요합니다. 일반적으로 이는 Snowflake에서 상위 권한 역할을 가진 데이터 또는 시스템 관리자가 처리합니다.


## Data Exchange을 통한 비공개 공유 소개 📓

[Data Exchange](https://docs.snowflake.com/ko/user-guide/data-exchange)는 초대한 특정 멤버들과 데이터를 안전하게 공유하고 협업할 수 있는 데이터 허브를 제공합니다. 공급자로서 데이터를 게시할 수 있으며, 교환에 참여하는 소비자가 이를 탐색할 수 있습니다.

Data Exchange를 이용하면 회사 내부 부서나 외부 벤더, 공급업체, 파트너 등 Data Exchange에 참여하는 일정한 비즈니스 파트너 그룹에게 데이터를 쉽게 제공할 수 있습니다. 조직 내외의 다양한 소비자들과 데이터를 공유하고 싶은 경우, 특정 소비자를 대상으로 하거나 Snowflake Marketplace에서 공개적으로 제공되는 Listing을 활용할 수도 있습니다.


### 비공개 공유 활용하기 🥋 

'공개' Snowflake Marketplace 외에도, 선별된 Snowflake 계정 또는 계정 내 사용자와 데이터 및 데이터 제품을 공유할 수 있습니다. 이 기능을 **Data Exchange**라고 합니다.

1. 왼쪽 메인 메뉴에서 **Horizon Catalog** > **Data sharing** > **External sharing** > **Shared With You**를 선택합니다.
1. 우리 계정과 공유된 Listing을 확인합니다. 대부분은 Snowflake Education Services에서 제공됩니다.
1. **Alpine Peaks Publishing** Listing을 클릭하여 내용을 검토합니다.

Snowflake 계정에서 이 데이터 세트를 대상으로 SQL 쿼리를 작성하여, **원예** 관련 최고의 도서를 찾아냅니다. 배움은 끝이 없습니다.

💡 **참고**: Snowflake Education Services에서 이미 우리 계정에 이 비공개 교환 데이터 세트를 확보했습니다. 파란색 **Get** 버튼이나 **Get Access** 링크를 클릭할 필요가 **없습니다**.

--- 

![Snowflake Data Exchange(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_pde_alpine_peaks_listing_1_v2.png)

- 이 공유 데이터를 본인의 역할에서 액세스할 수 있게 하려면 다음 SQL 구문을 실행하여 적절한 권한을 부여하세요.

- 오브젝트 브라우저를 새로 고치면 `ALPINE_PEAKS_PUBLISHING` 데이터베이스가 표시됩니다.


In [ ]:
CALL common_db.resources.access_alpine_peaks_publishing_data('{{user}}_learner_rl');

### 비공개 Data Exchange 데이터를 쿼리하세요. 🥋

Data Exchange의 **Alpine Peaks Publishing** Listing에는 다음과 같은 설명이 나와 있습니다.

- _Alpine Peaks Publishing은 가상의 출판사로, Snowflake Education에 2025년 신간 카탈로그를 공개했습니다. 이 컬렉션에는 다양한 장르의 여러 제목이 포함되어 있어 폭넓은 독자층을 대상으로 합니다. 비록 가상이지만 유명한 작가들의 작품을 선보이는 이 라인업은 모든 독자가 즐길 수 있는 콘텐츠를 제공합니다._

다음 SQL 셀에서 쿼리를 실행하여 이 데이터 세트를 탐색해 보세요.


In [ ]:
-- 이 데이터 세트에 몇 개의 책 리뷰가 있습니까?
SELECT COUNT(*) 
FROM alpine_peaks_publishing.books.forthcoming_releases;

In [ ]:
-- 이 데이터의 일부를 살펴보세요
SELECT * 
FROM alpine_peaks_publishing.books.forthcoming_releases
LIMIT 10;

In [ ]:
-- 이 데이터 세트에서 고유한 카테고리 수를 확인하세요
SELECT DISTINCT category 
FROM alpine_peaks_publishing.books.forthcoming_releases
ORDER BY 1;

In [ ]:
-- 카테고리별 총 책 수를 계산하세요
SELECT category, COUNT(*) 
FROM alpine_peaks_publishing.books.forthcoming_releases
GROUP BY category 
ORDER BY 2 DESC;

In [ ]:
-- 'garden'이 이름에 포함된 책 찾기
SELECT *
FROM alpine_peaks_publishing.books.forthcoming_releases
WHERE LOWER(title) LIKE '%garden%';

In [ ]:
-- 우리가 좋아하는 카테고리 'gardening'의 책을 나열하고 2025년에 출간될 날짜 순으로 정렬하세요
SELECT *
FROM alpine_peaks_publishing.books.forthcoming_releases
WHERE CONTAINS(LOWER(category),'garden')
ORDER BY release_date;


## Lesson 6 마무리 🏁 

### Lesson 6을 마무리할 준비가 되셨나요? 🏁 

- **ROOT_DEPTH** 테이블에 3개의 로우가 있나요? 

- **VEGETABLE_DETAILS** 테이블에 41개의 로우가 있나요?

- 두 테이블 모두 **(animal)_GARDEN_PLANTS** 데이터베이스의 **VEGGIES** 스키마에 있나요? 

이 모든 항목에 '예'라고 답했다면 이 수업을 완료로 표시하세요. 그렇지 않다면 돌아가서 잘못된 부분을 수정해야 합니다.


## 지식 테스트 :mag_right:

아래의 대화형 퀴즈 문제를 통해 이해도를 확인해 보세요. 각 `RUN_THIS_QUIZ_QUESTION_` 셀에는 Snowflake 기능과 관련된 객관식 문제를 제시하는 Streamlit 위젯이 포함되어 있습니다.  

**지침:**  
1. 노트북 셀 위에 커서를 올려 추가 컨트롤을 표시하세요.
1. 각 퀴즈 셀 오른쪽의 ▶️ **Play 버튼**을 클릭하여 실행하세요.  
1. 제공된 옵션에서 답을 선택하세요.  
1. 다음으로 넘어가기 전에 피드백을 검토하세요. 

💡 **참고:** 궁금하시면 셀을 확장하여 코드를 볼 수 있지만, 필수는 아닙니다. 이 퀴즈들은 필수 사항이 아닙니다. 배운 내용을 복습하며 연습할 기회를 제공하기 위한 것입니다.  


In [ ]:
st.divider()
question = "File Format이란 무엇인가요?"
options = ["아래 선택을 고르세요...",
           "A) 데이터가 도착할 때 Snowflake에 데이터 구조를 알리는 방법입니다", 
           "B) 테이블의 여백과 글꼴을 설정하는 방법입니다", 
           "C) 웹 페이지의 배경색을 선택하는 방법입니다"]
           
user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...": # 이 옵션은 streamlit이 1.26.0 이상으로 업그레이드될 때까지의 임시 방편입니다. 그래서 index=None을 사용할 수 있습니다.
        ''
    else:
        answer = 'b7d2bc171ddda1c19b0edb9e04f99d92'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

## 다음 단계

실습 단계를 완료하고 **지식 테스트** 질문에 정답을 입력하셨다면, Snowflake 강사의 안내에 따라 다음 Notebook으로 진행하세요.
